# Node 5 — knowledge indexing: PDF → chunks → embeddings → Chroma → answer

Exercises the whole knowledge-base feature against a real municipal PDF, one stage at a
time: extract the text, resolve its municipal metadata from the scrapper CSVs, split it
into overlapping windows, embed those windows with the multilingual retrieval model,
persist them in Chroma, query them back, and finally generate a cited answer.

Everything here is **real** — the actual `SentenceTransformerEmbedder`, the actual
persistent `ChromaVectorStore` at `Settings.chroma_path`, the actual chat model. The
unit tests in `tests/knowledge/` cover the same code with 2- and 3-dimensional stub
vectors and an in-memory store; this notebook is where you see what the chunk windows
and similarity scores actually look like on Spanish legal prose.

Sections 1-8 drive the services directly. Section 9 then runs the real
`KnowledgeIndexingNode` — the pipeline node itself, with its audit trail, its SSE
events and its `documents` catalogue row — including the failure path that keeps an
accepted document accepted when the vector store is down.

> **Kernel**: select the project's `.venv` kernel in the top-right picker.
>
> **First run downloads a model**: `paraphrase-multilingual-MiniLM-L12-v2` is ~470 MB.
> Section 8 additionally loads the chat model (a local GGUF at
> `Settings.chat_model_path`, or the Anthropic API if `CHAT_LLM_PROVIDER=claude`).
>
> **This writes to your real vector store** (`data/chroma` by default). Section 10
> removes everything this notebook added via `delete_by_job`, leaving the collection
> exactly as it was found — run the notebook through to the end so it cleans up after
> itself.

## 1 — Setup: the sample PDF, its hash, its job id

`playground/samples/` is gitignored, so the corpus lives only on your machine — grab it
from the Drive folder linked in `CLAUDE.md`, or produce it with
`uv run python scrapper/downloader.py`.

The filename matters: `CsvDocumentMetadataRepository` parses `{prefix}_{numero}_{anio}.pdf`
back into a CSV row in section 3, so `convenio_2_2013.pdf` resolves to real municipal
metadata while an arbitrary name degrades to filename-only.

`sha256` and `job_id` are the two keys everything downstream hangs off: chunk ids are
derived from the hash (which is what makes re-indexing idempotent), and `job_id` is what
section 10 deletes by.

In [ ]:
import hashlib
from pathlib import Path
from uuid import uuid4

import classiflow
from classiflow.settings import Settings

_SAMPLES_DIR = Path(classiflow.__file__).parent / "playground" / "samples"
_PDF_NAME = "boletin_2061_2026.pdf"
_PDF_PATH = _SAMPLES_DIR / _PDF_NAME

assert _PDF_PATH.exists(), (
    f"Missing {_PDF_NAME}. Drop a municipal PDF into {_SAMPLES_DIR} -- the folder is "
    "gitignored, so it is empty on a fresh clone. Any file named "
    "'{prefix}_{numero}_{anio}.pdf' resolves to CSV metadata in section 3."
)

pdf_bytes = _PDF_PATH.read_bytes()
sha256 = hashlib.sha256(pdf_bytes).hexdigest()
job_id = f"playground-node5-{uuid4().hex[:8]}"

print(f"file           : {_PDF_NAME}  ({len(pdf_bytes):,} bytes)")
print(f"sha256         : {sha256}")
print(f"job_id         : {job_id}\n")
print(f"chroma_path    : {Settings.chroma_path}")
print(f"collection     : {Settings.chroma_collection}")
print(f"embedding_model: {Settings.embedding_model}")
print(f"chunk_size     : {Settings.chunk_size}  overlap: {Settings.chunk_overlap}")
print(f"retrieval_top_k: {Settings.retrieval_top_k}")
print(f"chat provider  : {Settings.chat_llm_provider}")

AssertionError: Missing convenio_2_2013.pdf. Drop a municipal PDF into C:\Repos\Diplo-TP-Final\Trabajo-Integrador\Trabajo-Integrador\src\classiflow\playground\samples -- the folder is gitignored, so it is empty on a fresh clone. Any file named '{prefix}_{numero}_{anio}.pdf' resolves to CSV metadata in section 3.

## 2 — Extract the text

The same `TextExtractor` chain the pipeline uses: MarkItDown first, falling through to
EasyOCR only when the result is below `MIN_TEXT_FOR_OCR`.

Which path runs depends on the document you loaded in section 1. A PDF with a real text
layer (most `convenio_*` / `ordenanza_*` files) is handled by MarkItDown in seconds; a
genuine scan with no text layer — common for older `boletin_*` files — falls through to
OCR, which renders every page at `Settings.ocr_render_dpi` and takes minutes on CPU. See
`text_extraction.ipynb` for both paths in isolation.

Whatever comes out here is exactly what node 5 receives from the coordinator as
`state.get("text", "")`.

In [ ]:
import easyocr

from classiflow.ingesta.extract import MIN_TEXT_FOR_OCR, TextExtractor
from classiflow.ingesta.extractors import MarkItDownExtractor, OCRExtractor

text_extractor = TextExtractor([
    MarkItDownExtractor(),
    OCRExtractor(reader=easyocr.Reader([Settings.ocr_lang], gpu=True)),
])

text = text_extractor(pdf_bytes, _PDF_NAME)

print(f"extracted: {len(text):,} chars  (MIN_TEXT_FOR_OCR = {MIN_TEXT_FOR_OCR})")
print(f"preview  : {text[:300]!r}")
assert text, "Extraction returned nothing -- there would be no chunks to index."

## 3 — Resolve the municipal metadata

`CsvDocumentMetadataRepository` maps the filename back to a row in `scrapper/*.csv`,
which is where the document's número, año, subject and — most usefully for the chat
UI — its public `download_url` come from.

Every field except `filename` is best-effort by design: a manual upload that matches no
CSV row still yields a valid `DocumentMetadata`, so retrieval keeps working and the
source simply has no link. The repository logs and degrades; it never raises.

In [ ]:
from classiflow.knowledge.infrastructure.csv_metadata import CsvDocumentMetadataRepository

metadata_repo = CsvDocumentMetadataRepository()
metadata = metadata_repo.resolve(_PDF_NAME)

print(f"citation: {metadata.citation}\n")
for field, value in metadata.model_dump().items():
    print(f"  {field:<19}: {value!r}")

unmatched = metadata_repo.resolve("manual_upload.pdf")
print(f"\nunmatched filename -> citation={unmatched.citation!r}, url={unmatched.download_url!r}")

## 4 — Chunk it

`ChunkerService` splits on paragraph breaks and packs them into windows of at most
`Settings.chunk_size`; a single paragraph longer than the window (common in scanned norms
with no blank lines) is hard-split on a fixed stride with `chunk_overlap` characters of
overlap.

Note the header line prepended to every chunk: retrieval returns single chunks, and a
bare fragment of legal prose is often unattributable on its own, so each one carries
`"{citation} — {subject}"` as its first line.

Chunk ids are `f"{sha256}:{index}"` — deterministic, not random. That is the whole
mechanism behind the idempotent re-index in section 6.

In [ ]:
from classiflow.knowledge.chunker import ChunkerService
from classiflow.knowledge.domain.chunk import Chunk

chunker = ChunkerService()
chunks = chunker.split(text, job_id=job_id, sha256=sha256, metadata=metadata)
assert chunks, "No chunks produced -- nothing would reach the vector store."

lengths = [len(chunk.text) for chunk in chunks]
print(f"chunks : {len(chunks)}")
print(f"lengths: min={min(lengths)} max={max(lengths)} mean={sum(lengths) // len(lengths)}")
print(f"chunk_id[0] == make_id(sha256, 0): {chunks[0].chunk_id == Chunk.make_id(sha256, 0)}")

for chunk in chunks[:2]:
    print(f"\n--- {chunk.chunk_id}  (index {chunk.chunk_index}) ---")
    print(chunk.text[:500])

## 5 — Embed the chunks

`SentenceTransformerEmbedder` encodes with `normalize_embeddings=True`, so every vector
is unit length and the cosine similarity Chroma computes reduces to a dot product.

This is deliberately **not** node 4's model. Duplicate control uses `all-MiniLM-L6-v2`
with a cosine threshold calibrated in `config/duplicate_control.yaml`; swapping it there
would invalidate that threshold. Retrieval instead uses a multilingual model, because
the corpus is Spanish.

In [ ]:
import math

from classiflow.knowledge.infrastructure.embedder import SentenceTransformerEmbedder

_NORM_TOLERANCE = 1e-3

embedder = SentenceTransformerEmbedder()
vectors = embedder.embed_documents([chunk.text for chunk in chunks])

norm = math.sqrt(sum(value * value for value in vectors[0]))
print(f"model      : {embedder.model_name}")
print(f"vectors    : {len(vectors)} x {len(vectors[0])} dims")
print(f"L2 norm [0]: {norm:.6f}")
assert abs(norm - 1.0) < _NORM_TOLERANCE

## 6 — Store them in Chroma

`IndexerService.index(...)` is the single call node 5 makes, and it does everything
sections 3-5 just did by hand: resolve metadata, chunk, embed, upsert. Embedding and the
Chroma write are both blocking, so it pushes them onto a worker thread with
`asyncio.to_thread` — indexing runs inside a live pipeline job with other jobs and SSE
streams in flight.

Chroma is opened lazily on first use, as a `PersistentClient` over `Settings.chroma_path`
with `hnsw:space = "cosine"`. Embeddings are always supplied by the caller; Chroma's own
embedding function is never used, so the embedder port stays the one place that decides
which model produces vectors.

In [ ]:
from classiflow.knowledge.indexer import IndexerService
from classiflow.knowledge.infrastructure.chroma_store import ChromaVectorStore

store = ChromaVectorStore()
baseline_count = store.count()

indexer = IndexerService(
    chunker=chunker,
    embedder=embedder,
    vector_store=store,
    metadata_repo=metadata_repo,
)

result = await indexer.index(job_id, _PDF_NAME, sha256, text)

print(f"collection count before: {baseline_count}")
print(f"chunks indexed         : {result.chunk_count}")
print(f"collection count after : {store.count()}")
print(f"resolved citation      : {result.metadata.citation}")

### 6b — Re-indexing overwrites in place; blank text is a no-op

Because chunk ids are derived from the document hash, indexing the same document twice
upserts over the same rows instead of duplicating them — that is what makes a re-index
after a vector-store outage safe to just run again.

A document with no usable text short-circuits before chunking and writes nothing at all.

In [ ]:
after_first = store.count()
repeated = await indexer.index(job_id, _PDF_NAME, sha256, text)

print(f"re-indexed chunks: {repeated.chunk_count}")
print(f"count after      : {store.count()}  (was {after_first})")
assert store.count() == after_first

blank = await indexer.index(job_id, _PDF_NAME, sha256, "   ")
print(f"blank text       -> chunks={blank.chunk_count}, count still {store.count()}")

## 7 — Query it back

`RagService.retrieve` embeds the question with the *same* model, asks Chroma for the
top-k nearest chunks, and converts Chroma's cosine **distance** into the similarity
score callers actually want (`1.0 - distance`). Scores come back descending.

`top_k` defaults to `Settings.retrieval_top_k`; a `ChatQuery` can override it per call.

In [ ]:
from collections.abc import AsyncIterator

from classiflow.knowledge.domain.chat import ChatQuery
from classiflow.knowledge.infrastructure.claude_chat_llm import ClaudeChatLlm
from classiflow.knowledge.infrastructure.llama_chat_llm import LlamaCppChatLlm
from classiflow.knowledge.rag import RagService


class _OfflineChatLlm:
    # Stand-in for when neither chat provider is usable on this machine. Retrieval,
    # prompt building and source resolution all stay real -- only the generated text is
    # replaced, so sections 8 and 8b still exercise the whole RAG path end to end.
    def __init__(self) -> None:
        self.last_prompt = ""

    async def astream(self, system: str, user: str) -> AsyncIterator[str]:
        self.last_prompt = user
        yield "[sin modelo de chat configurado] "
        yield f"el prompt se construyo bien: system={len(system)} chars, "
        yield f"user={len(user)} chars con los pasajes recuperados."


_gguf_path = Path(Settings.chat_model_path)
_has_llama = _gguf_path.exists()
_has_claude = bool(Settings.anthropic_api_key)

# Prefer the configured provider, fall back to whichever is actually usable, and only
# then to the stub -- an absent 2.5 GB GGUF should not break the other nine sections.
if Settings.chat_llm_provider == "claude" and _has_claude:
    chat_llm, provider_note = ClaudeChatLlm(), f"Anthropic API ({Settings.anthropic_model})"
elif _has_llama:
    chat_llm, provider_note = LlamaCppChatLlm(), f"local GGUF ({_gguf_path.name})"
elif _has_claude:
    chat_llm, provider_note = ClaudeChatLlm(), f"Anthropic API ({Settings.anthropic_model})"
else:
    chat_llm, provider_note = _OfflineChatLlm(), "offline stub -- no chat model configured"

rag = RagService(embedder=embedder, vector_store=store, chat_llm=chat_llm)

query = ChatQuery(question=f"¿Cuáles son los puntos principales de {metadata.citation}?")
retrieved = await rag.retrieve(query)

print(f"chat provider: {provider_note}")
print(f"question     : {query.question}\n")
for hit in retrieved:
    print(f"score={hit.score:.4f}  {hit.chunk_id}")
    print(f"    {hit.text[:180]!r}\n")

### 7b — Metadata filters

Chunk metadata is stored flat (Chroma only accepts scalars), which is exactly what its
`where` clause can match on. Filtering narrows the candidate set *before* the nearest-
neighbour search — this is how the chat UI will scope a question to one document type or
one year.

In [ ]:
filtered = await rag.retrieve(
    ChatQuery(question=query.question, filters={"doc_type": metadata.doc_type})
)
missing = await rag.retrieve(ChatQuery(question=query.question, filters={"doc_type": "NoSuchType"}))

print(f"filters={{'doc_type': {metadata.doc_type!r}}} -> {len(filtered)} hits")
print(f"filters={{'doc_type': 'NoSuchType'}} -> {len(missing)} hits")

## 8 — Generate an answer

`RagService.answer` retrieves, builds the prompt from the passages, and runs the chat
model. The system prompt is Spanish and instructs the model to cite by tipo/número/año
and to say so when the passages do not contain the answer.

Section 7 picked the provider and printed which one is in play. Everything except the
generated text — retrieval, prompt construction, source resolution — is identical in all
three cases, so this section is meaningful even without a model installed:

| Provider | What it needs |
|---|---|
| `llama` (default) | A GGUF at `Settings.chat_model_path`, i.e. `src/classiflow/ingesta/models/Phi-4-mini-instruct-Q4_K_M.gguf` (~2.5 GB, not in the repo). Loaded at `n_ctx = CHAT_MODEL_N_CTX` (8192 — passages plus a question do not fit in the 2048 the validation nodes use). Fully offline, slow on CPU. |
| `claude` | `ANTHROPIC_API_KEY` in `.env`, plus `CHAT_LLM_PROVIDER=claude`. |
| offline stub | Nothing. Used automatically when neither of the above is available. |

**To get a real answer here**, pick either:
- **Anthropic** — set `ANTHROPIC_API_KEY=sk-ant-...` and `CHAT_LLM_PROVIDER=claude` in
  `.env`, then restart the kernel. Fastest route, no download.
- **Local GGUF** — download `Phi-4-mini-instruct-Q4_K_M.gguf` into
  `src/classiflow/ingesta/models/` (the directory does not exist yet; create it). This
  is the same model node 3's legitimacy check uses, so it unblocks that too.

In [ ]:
answer = await rag.answer(query)

print(answer.answer)
print("\nsources:")
for source in answer.sources:
    label = f"{source.doc_type} {source.number}/{source.year}".strip()
    print(f"  [{source.score:.4f}] {label or source.filename}")
    if source.download_url:
        print(f"            {source.download_url}")

### 8b — Streaming

`astream` resolves the sources *before* generation starts and repeats the same list with
every token, so a UI can render citations immediately instead of waiting for the answer
to finish.

How many chunks you actually see depends on the provider: `claude` streams real tokens;
the local llama.cpp provider yields the whole completion as a single chunk (token-level
streaming would need a queue bridging its worker thread back to the event loop, not worth
it for the offline fallback); the stub yields three, just to make the mechanism visible.

In [ ]:
streamed_sources: list[object] = []
async for token, sources in rag.astream(query):
    streamed_sources = sources
    print(token, end="")

print(f"\n\nsources carried alongside the stream: {len(streamed_sources)}")

## 9 — The node itself: audit trail, SSE events, catalogue row

Everything above used the services directly. `KnowledgeIndexingNode` is what the
coordinator actually runs after node 4 on the accept path, and it adds three things the
services do not: a broadcast `started`/`passed` event pair, an audit record, and a row in
the `documents` table.

That table is the catalogue — it holds no vectors (those live in Chroma on disk), it is
the source of truth for *rebuilding* the collection if the Chroma directory is ever lost.

Uses its own database file rather than the shared dev one, following
`pipeline_benchmark.ipynb`.

In [ ]:
import asyncio

from sqlalchemy import select
from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine

from classiflow.database.base import Base
from classiflow.database.models import Document
from classiflow.database.repositories.audit import SqlAuditRepository
from classiflow.database.repositories.document import SqlDocumentRepository
from classiflow.events.broadcaster import EventBroadcaster
from classiflow.ingesta.domain.context import JobContext
from classiflow.ingesta.nodes.node5_knowledge_indexing import KnowledgeIndexingNode
from classiflow.services.audit.service import AuditService

_db_path = Path(classiflow.__file__).parents[2] / "data" / "node5_playground.db"
_db_path.parent.mkdir(parents=True, exist_ok=True)

engine = create_async_engine(f"sqlite+aiosqlite:///{_db_path.as_posix()}", echo=False)
session_factory = async_sessionmaker(engine, expire_on_commit=False)

async with engine.begin() as conn:
    await conn.run_sync(Base.metadata.create_all)

print(f"catalogue database: {_db_path}")

In [ ]:
node_job_id = f"{job_id}-node"
node_events: list[object] = []
broadcaster = EventBroadcaster()


async def _collect_events() -> None:
    async for event in broadcaster.subscribe(node_job_id):
        node_events.append(event)
        print(f"  SSE -> node={event.node}  status={event.status}")


async with session_factory() as session:
    node = KnowledgeIndexingNode(
        audit=AuditService(SqlAuditRepository(session)),
        broadcaster=broadcaster,
        indexer=indexer,
        document_repo=SqlDocumentRepository(session),
    )
    collect_task = asyncio.create_task(_collect_events())
    await asyncio.sleep(0)  # let the subscriber attach before run() emits

    node_result = await node.run(JobContext(job_id=node_job_id, filename=_PDF_NAME), sha256, text)

    await broadcaster.close(node_job_id)
    await collect_task
    await session.commit()

print(f"\npassed      : {node_result.passed}")
print(f"indexed     : {node_result.indexed}")
print(f"chunk_count : {node_result.chunk_count}")
print(f"citation    : {node_result.doc_type} {node_result.number}/{node_result.year}")
print(f"download_url: {node_result.download_url}")

### 9a — The catalogue row it persisted

In [ ]:
async with session_factory() as session:
    row = (await session.execute(select(Document).where(Document.sha256 == sha256))).scalar_one()

print(f"id             : {row.id}")
print(f"job_id         : {row.job_id}")
print(f"filename       : {row.filename}")
print(f"doc_type/num/yr: {row.doc_type} {row.number}/{row.year}")
print(f"subject        : {row.subject}")
print(f"chunk_count    : {row.chunk_count}")
print(f"indexed_at     : {row.indexed_at}")

### 9b — A vector-store outage must not un-accept the document

This is node 5's defining behaviour. The document has already cleared file reception,
format validation, content validation and duplicate control — a knowledge-base failure
arriving *after* all of that must not retroactively reject it.

So `_index` catches `KnowledgeError` and returns `passed=True, indexed=False` with the
cause recorded in the audit log, and writes no catalogue row. The coordinator reinforces
this from the other side: `KnowledgeIndexingResult` is deliberately excluded from the
union `_get_rejection_reason` inspects, so an indexing failure can never become the job's
rejection reason. Re-index later from the catalogue.

In [ ]:
from classiflow.knowledge.exceptions import VectorStoreError
from classiflow.knowledge.repositories.vector_store import Embedding

_BROKEN_SHA256 = "f" * 64


class _BrokenVectorStore(ChromaVectorStore):
    def upsert(self, chunks: list[Chunk], _embeddings: list[Embedding]) -> None:
        self.attempted = len(chunks)
        raise VectorStoreError(operation="upsert", cause="disk unavailable")


broken_indexer = IndexerService(
    chunker=chunker,
    embedder=embedder,
    vector_store=_BrokenVectorStore(),
    metadata_repo=metadata_repo,
)

async with session_factory() as session:
    broken_node = KnowledgeIndexingNode(
        audit=AuditService(SqlAuditRepository(session)),
        broadcaster=EventBroadcaster(),
        indexer=broken_indexer,
        document_repo=SqlDocumentRepository(session),
    )
    broken_result = await broken_node.run(
        JobContext(job_id=f"{job_id}-broken", filename=_PDF_NAME), _BROKEN_SHA256, text
    )
    await session.commit()

    orphan = (
        await session.execute(select(Document).where(Document.sha256 == _BROKEN_SHA256))
    ).scalar_one_or_none()

print(f"passed               : {broken_result.passed}")
print(f"indexed              : {broken_result.indexed}")
print(f"rejection_reason     : {broken_result.rejection_reason}")
print(f"catalogue row written: {orphan is not None}")
assert broken_result.passed
assert not broken_result.indexed

## 10 — Clean up

Removes every chunk this notebook added, so your real Chroma collection is left exactly
as it was found — the two job ids below are the only things it wrote into the shared
store. Needs section 9 to have run, since that is where `node_job_id` is defined.

Worth knowing why the count still lands back on the baseline: section 9 re-indexed the
*same* document through the node, and chunk ids are derived from the hash, so those rows
overwrote section 6's in place and now carry `node_job_id`. Deleting both ids covers
either outcome.

`data/node5_playground.db` is left on disk for inspection; delete it whenever you like.

In [ ]:
for stale_job in (job_id, node_job_id):
    store.delete_by_job(stale_job)

print(f"count after cleanup: {store.count()}  (baseline was {baseline_count})")
assert store.count() == baseline_count

await engine.dispose()
print("engine disposed")